# 05 — Profundización de PCA: Robustez Multi-Semilla, Frente de Pareto y Atribución Espacial Multiescala


---

## Objetivo
1. Varianza retenida (87.4% en 64 componentes).
2. Estabilidad (5 semillas): media y dispersión en 5 semillas; qué fórmula de 1 variable aparece ($x_1 = \text{PC}_0$).
3. Pareto (variables vs AUC): Barrido de aristas fijadas en capa 0 ($k \in [4, 8, 12, 16, 20, 24, 32, 48, \text{all}]$).
4. Mapa inverso: Identificación de los 10 canales convolucionales de `layer4` de ResNet18 que constituyen $x_1$.
5. Atribución (Grad-CAM): Dónde cae Grad-CAM de la fórmula en CBIS-DDSM.
6. Salidas: Guarda tablas .tex y figuras.


In [ ]:
import os, sys, time, json, re
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from PIL import Image

for p in (os.environ.get("TESIS_CODIGOS", os.path.abspath("..")), os.environ.get("TESIS_REP", os.path.abspath(os.path.join("..", "artifacts")))):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.explainability.f2_symb import SymbHead, softmax2, youden_threshold
from src.explainability.pca_attribution import PCAGradCAM, get_top_pca_channels
from src.data.preprocessing import get_val_transforms
from torchvision import models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

In [ ]:
import os
EMB = os.environ.get("TESIS_EMBEDDINGS_DIR", os.path.abspath(os.path.join("..", "artifacts", "output", "embeddings")))
feats512, labs512 = {}, {}
for split in ("train", "val", "test"):
    feats512[split] = np.load(os.path.join(EMB, f"f2base_{split}_feats.npy")).astype(np.float32)
    labs512[split] = np.load(os.path.join(EMB, f"f2base_{split}_labels.npy")).astype(int)

mu512 = feats512["train"].mean(0, keepdims=True)
sd512 = feats512["train"].std(0, keepdims=True) + 1e-8
feats512_z = {s: (feats512[s] - mu512) / sd512 for s in ("train", "val", "test")}
print(f"Train: {feats512['train'].shape} | Val: {feats512['val'].shape} | Test: {feats512['test'].shape}")

## 1. Análisis Espectral de PCA (Justificación Dimensional)


In [3]:
pca_full = PCA(n_components=128, random_state=42)
pca_full.fit(feats512_z["train"])
var_cum = np.cumsum(pca_full.explained_variance_ratio_) * 100

for k in [16, 24, 32, 48, 64]:
    print(f"k={k:2d} componentes -> Varianza explicada: {var_cum[k-1]:.2f}%")

k=16 componentes -> Varianza explicada: 73.25%
k=24 componentes -> Varianza explicada: 77.88%
k=32 componentes -> Varianza explicada: 80.92%
k=48 componentes -> Varianza explicada: 84.82%
k=64 componentes -> Varianza explicada: 87.47%


## 2. Batería Multi-Semilla de Robustez (5 Semillas)


In [ ]:
# Resultados cargados del estudio multi-semilla ejecutado
import os
with open(os.path.join(os.environ.get("TESIS_REP", os.path.abspath(os.path.join("..", "artifacts"))), "output", "frente2", "n22_pca_deepening.json"), "r", encoding="utf-8") as f:
    n22_data = json.load(f)

df_seeds = pd.DataFrame(n22_data["multi_seed"])
df_seeds[["seed", "auc_full", "D_full_test", "D_cmp_test", "r_C_full", "recall_D_full", "spec_D_full", "n_vars_full"]]

## 3. Frontera de Pareto: Complejidad Simbólica vs. Rendimiento


In [5]:
df_pareto = pd.DataFrame(n22_data["pareto_frontier"])
df_pareto[["top_k", "n_vars", "n_chars", "auc_D", "recall", "spec"]]

,top_k,n_vars,n_chars,auc_D,recall,spec
0,4,1,82,0.6621,0.8062,0.3754
1,8,1,148,0.6621,0.8062,0.3754
2,12,1,148,0.7237,0.6603,0.6549
3,16,2,172,0.7202,0.6531,0.6650
4,20,2,172,0.7202,0.6531,0.6650
5,24,2,172,0.7200,0.6603,0.6448
6,32,2,172,0.7200,0.6603,0.6448
7,48,2,196,0.7111,0.6435,0.6919
8,all,16,880,0.7365,0.6077,0.7290


## 4. Mapa inverso: $PC_0 \to$ Canales Convolucionales de ResNet18


In [6]:
df_ch = pd.DataFrame(n22_data["top_channels_pc0"])
df_ch

,channel,weight,abs_weight
0,96,0.082327,0.082327
1,280,0.082186,0.082186
2,360,0.081976,0.081976
3,408,0.081643,0.081643
4,232,0.081622,0.081622
5,467,0.081350,0.081350
6,367,0.081187,0.081187
7,263,0.081020,0.081020
8,54,0.080810,0.080810
9,155,0.079964,0.079964


## 5. Atribución Espacial Multiescala (Grad-CAM de $PC_0$)


In [ ]:
# Visualización generada de atribución espacial
import os
img_cam = Image.open(os.path.join(os.environ.get("TESIS_CODIGOS", os.path.abspath("..")), "reports", "figures", "22_pca_gradcam_atribucion.png"))
plt.figure(figsize=(10, 7))
plt.imshow(img_cam)
plt.axis("off")
plt.show()

## 6. Cierre
- PCA: Retiene 87.41% de varianza.
- Estabilidad: Media de AUC_D = 0.7257 +/- 0.0070, frente al baseline ResNet18 (0.7325).
- x1: La variable $x_1$ ($PC_0$) aparece en la formula compacta en las 5 semillas.
- Espacio: Los términos de la fórmula se proyectan a la activación de layer4 en la lesión.
